In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Adjust the path to point to external/AlphaPEM
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.sampling.sampler import get_polarisation_curve_samples, build_fixed_parameters
from src.analysis.sensitivity import SensitivityAnalyzer
from omegaconf import OmegaConf
from scipy.stats.qmc import LatinHypercube

In [2]:
from sklearn.model_selection import train_test_split
data = pd.read_pickle('../sampling_test/validated_final_df_57344_imputed.pkl')
data = data[data['Ucell'].apply(lambda x: all(val > 0 for val in x))]

exploaded_df = data.explode(['ifc', 'Ucell'])

np.random.seed(42)

parameters = ['Tfc', 'i0_c_ref', 'Pa_des', 'kappa_c', 'kappa_co', 'tau', 'Re', 'ifc']

# Get unique curve IDs and split train/test
unique_ids = data['SHA256'].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

exploaded_df = exploaded_df[exploaded_df['SHA256'].isin(test_ids)]

In [4]:
import os
import pandas as pd

def load_all_pysr_equations(base_dir="../models/pysr_equations/", output_csv="all_equations_combined.csv"):
    """
    Loop through subfolders in a given directory, read each equations.csv file,
    and combine all equations into a single DataFrame.

    Args:
        base_dir (str): Path to the directory containing PySR model subfolders.
        output_csv (str): Output CSV path to save the combined equations.
    
    Returns:
        pd.DataFrame: Combined DataFrame of all equations.
    """
    all_equations = []

    for subdir in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, subdir)
        equations_file = os.path.join(folder_path, "hall_of_fame.csv")

        if os.path.isdir(folder_path) and os.path.isfile(equations_file):
            df = pd.read_csv(equations_file)
            df["model_folder"] = subdir  # track which folder it came from
            all_equations.append(df)

    if not all_equations:
        print("No equations.csv files found.")
        return pd.DataFrame()

    combined_df = pd.concat(all_equations, ignore_index=True)
    if output_csv is not None:
        combined_df.to_csv(output_csv, index=False)
        print(f"Saved combined equations to {output_csv}")
    return combined_df

df_all_equations = load_all_pysr_equations(output_csv=None)
df_all_equations = df_all_equations[df_all_equations['model_folder'] == '20250727_122629']  # Example check, adjust as needed

In [5]:
import re
import numpy as np
import pandas as pd
from sympy import sympify, symbols, lambdify
from sklearn.metrics import r2_score, mean_squared_error

import numpy as np
import pandas as pd
import re
from sympy import sympify, symbols, lambdify
from sklearn.metrics import r2_score, mean_squared_error

# Define NumPy-safe unary operators
numpy_locals = {
    'square': lambda x: np.power(x, 2),
    'cube': lambda x: np.power(x, 3),
    'inv': lambda x: 1 / x,
    'neg': lambda x: -x,
}

def evaluate_equations_with_parameters(df_equations, X, y, parameters):
    results = []
    X_df = pd.DataFrame(X, columns=parameters)

    for _, row in df_equations.iterrows():
        equation_str = str(row['Equation'])

        try:
            # Replace x0, x1, ... with parameter names
            for i, name in enumerate(parameters):
                equation_str = equation_str.replace(f'x{i}', name)

            # Parse without custom locals — just pure sympy parsing
            expr = sympify(equation_str)

            # If scalar (no variables)
            if len(expr.free_symbols) == 0:
                y_pred = np.full_like(y, float(expr), dtype=np.float64)
            else:
                variables = sorted([str(s) for s in expr.free_symbols])
                sym_vars = symbols(variables)

                # lambdify using numpy functions + custom ops
                f = lambdify(sym_vars, expr, modules=[numpy_locals, 'numpy'])

                inputs = [X_df[var].values for var in variables]
                y_pred = f(*inputs)

            y_pred = np.array(y_pred, dtype=np.float64)

            if np.any(np.isnan(y_pred)) or np.any(np.isinf(y_pred)):
                raise ValueError("Prediction contains NaN or Inf")

            r2 = r2_score(y, y_pred)
            mse = mean_squared_error(y, y_pred)

        except Exception as e:
            print(f"❌ Failed to evaluate equation: {row['Equation']}\n   Error: {e}")
            r2 = mse = np.nan

        results.append({
            'equation': row['Equation'],
            'r2': r2,
            'mse': mse
        })

    return pd.DataFrame(results)


df = exploaded_df[parameters].copy()
df['ifc'] = df['ifc'].astype(float)  # Ensure ifc is float for consistency
results = evaluate_equations_with_parameters(
    df_all_equations,
    df,
    exploaded_df['Ucell'],
    parameters
)

In [6]:
results.sort_values(by='r2', ascending=False, inplace=True)

In [7]:
for idx, row in results.iterrows():
    equation_str = row['equation']
    
    # Replace x0, x1, ... with parameter names in the string
    for i, name in enumerate(parameters):
        equation_str = equation_str.replace(f"x{i}", name)

    print(f"Equation: {equation_str}  R²: {row['r2']:.4f}, MSE: {row['mse']:.4f}")

Equation: ((kappa_c + log(i0_c_ref)) * ((Pa_des * 1.5976357e-7) + 0.019371508)) + ((5.5642056 - log(Tfc)) + exp(ifc * -0.19848849))  R²: 0.8538, MSE: 0.0042
Equation: ((kappa_c + log(i0_c_ref)) * (Pa_des * 2.0317603e-7)) + ((5.61315 - log(Tfc)) + exp(ifc * -0.19978261))  R²: 0.8450, MSE: 0.0044
Equation: ((kappa_c + log(i0_c_ref)) * (Pa_des * 2.0114945e-7)) + ((ifc * -0.1469058) - (log(Tfc) + -6.5887384))  R²: 0.8338, MSE: 0.0047
Equation: (((kappa_c - -22.400759) / Tfc) * log(Pa_des * i0_c_ref)) + ((ifc * -0.147784) + -0.124869585)  R²: 0.8305, MSE: 0.0048
Equation: (((kappa_c - -19.63474) / Tfc) * log(i0_c_ref * Pa_des)) + (ifc * -0.1474736)  R²: 0.8280, MSE: 0.0049
Equation: ((20.85901 / Tfc) * log(Pa_des * i0_c_ref)) + (ifc * -0.1474275)  R²: 0.7968, MSE: 0.0058
Equation: (ifc * -0.14658587) + (log(Pa_des * i0_c_ref) * 0.059901953)  R²: 0.7560, MSE: 0.0069
Equation: (i0_c_ref + 302.01355) / (Tfc + (ifc * 96.30289))  R²: 0.7386, MSE: 0.0074
Equation: inv(inv(i0_c_ref) + exp(ifc * 0.

In [8]:
from sympy import sympify, latex

def add_latex_column(results_df, parameters):
    latex_eqs = []

    for eq in results_df['equation']:
        # Replace x0, x1, ... with actual names
        for i, name in enumerate(parameters):
            eq = eq.replace(f"x{i}", name)

        try:
            expr = sympify(eq)
            latex_eqs.append(latex(expr))
        except Exception as e:
            latex_eqs.append(f"Invalid: {e}")

    results_df['latex'] = latex_eqs
    return results_df

results_df = add_latex_column(results, parameters)
print(results_df[['equation', 'latex']])


                                             equation  \
13  ((x3 + log(x1)) * ((x2 * 1.5976357e-7) + 0.019...   
12  ((x3 + log(x1)) * (x2 * 2.0317603e-7)) + ((5.6...   
11  ((x3 + log(x1)) * (x2 * 2.0114945e-7)) + ((x7 ...   
10  (((x3 - -22.400759) / x0) * log(x2 * x1)) + ((...   
9   (((x3 - -19.63474) / x0) * log(x1 * x2)) + (x7...   
8   ((20.85901 / x0) * log(x2 * x1)) + (x7 * -0.14...   
7   (x7 * -0.14658587) + (log(x2 * x1) * 0.059901953)   
6           (x1 + 302.01355) / (x0 + (x7 * 96.30289))   
5                 inv(inv(x1) + exp(x7 * 0.20674646))   
4                   exp((inv(x1) + x7) * -0.22105232)   
3                   inv((x7 * 0.2868409) + 1.0120518)   
2                                 exp(x7 * -0.229922)   
1                                      249.33754 / x0   
0                                          0.71649367   

                                                latex  
13  \left(1.5976357 \cdot 10^{-7} Pa_{des} + 0.019...  
12  2.0317603 \cdot 10^{-7} Pa_{

In [9]:
from IPython.display import display, Math

for _, row in results_df.head(30).iterrows():
    eq_latex = row['latex']
    r2 = row['r2']
    mse = row['mse']
    
    display(Math(
        rf"\text{{Equation: }} {eq_latex} \\ \text{{R}}^2 = {r2:.4f}, \quad \text{{MSE}} = {mse:.4f}"
    ))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>